# grad-accumulate-on-leaf — worked example 2: Manual accumulate_grad with three distinct contributions

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-accumulate-on-leaf`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `accumulate_grad` function is the atomic unit of PyTorch's gradient accumulation. It follows a simple two-branch rule: if the leaf has no gradient yet (`leaf.grad is None`), assign directly; otherwise add to what's already there using rebinding (`leaf.grad = leaf.grad + g`), not in-place `+=`. Using `+` instead of `+=` matters because any code that holds a reference to the old gradient tensor should not see it mutated unexpectedly.

## Worked solution

**Step 1 — define MiniTensor stand-in.** We use a simple namespace object to mimic the `.grad` attribute of a leaf parameter. Initially `.grad = None`.

**Step 2 — first call: first-touch path.** When `leaf.grad is None`, we set `leaf.grad = g1` directly. This avoids allocating a zero tensor just to add to it immediately.

**Step 3 — second call: accumulate path.** Now `leaf.grad` is not `None`, so we compute `leaf.grad = leaf.grad + g2`. The old gradient tensor is not mutated — a new tensor is created and bound to `leaf.grad`.

**Step 4 — third call: another accumulation.** Same pattern with `g3`. The final result should be `g1 + g2 + g3`, verifying that all three contributions landed.

**Step 5 — verify and show reference isolation.** We save a reference to `leaf.grad` before calling with `g3`, then confirm the saved reference was NOT modified by the call (because we used `+`, not `+=`).

In [ ]:
import torch as t

t.manual_seed(42)

class SimpleLeaf:
    def __init__(self):
        self.grad = None

def accumulate_grad(leaf, g):
    """Accumulate gradient g into leaf.grad using rebinding (not in-place)."""
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g  # rebind, don't mutate in place

# Three gradient contributions (as if from three backward paths)
g1 = t.tensor([1.0, 2.0, 3.0])
g2 = t.tensor([0.5, 0.5, 0.5])
g3 = t.tensor([2.0, 0.0, -1.0])

leaf = SimpleLeaf()

# First touch: should set directly
accumulate_grad(leaf, g1)
print(f"After 1st call: {leaf.grad}")  # [1.0, 2.0, 3.0]

# Save reference before 3rd call to test isolation
accumulate_grad(leaf, g2)
ref_before_g3 = leaf.grad
print(f"After 2nd call: {leaf.grad}")  # [1.5, 2.5, 3.5]

accumulate_grad(leaf, g3)
print(f"After 3rd call: {leaf.grad}")   # [3.5, 2.5, 2.5]

# Confirm total equals sum of all three
expected = g1 + g2 + g3
assert t.allclose(leaf.grad, expected), f"Expected {expected}, got {leaf.grad}"

# Reference isolation: ref_before_g3 must NOT have been changed in place
assert t.allclose(ref_before_g3, g1 + g2), "rebind violated: old ref was mutated!"
print("All checks passed — rebind correctly isolates old grad references.")